# The Edge of Stability

Where the escape threshold comes from - the dynamics, or the arithmetic.

Full specification: [`docs/edge-of-stability.md`](../docs/edge-of-stability.md)

The parent notebook ([`deep_linear_dynamics.ipynb`](deep_linear_dynamics.ipynb)) verified the
training loop at $\eta = 0.01$, deep inside the convergent regime. This one runs the same
system at step sizes large enough to break it, and asks what sets the point where it breaks.

`docs/deep-linear-dynamics.md` §7 already asserts an answer - the 5x5 network escapes at
$\eta \approx 1.266$ rather than the $2/s_{\max} = 2$ the scalar mode map predicts, and that
threshold is invariant to precision and to perturbation size. **The code that produced those
numbers is not in this repository and no longer exists.** They are unsupported claims, not
targets. Whatever this notebook measures replaces them.

Four claims, fixed in §3 of the protocol before any of this ran:

| | Claim | Type |
|---|---|---|
| **C1** | The $\eta$ sweep has three regimes: convergent, edge-of-stability, escaped | structure |
| **C2** | The threshold does not move with machine $\varepsilon$ across twelve orders | null |
| **C3** | Nor with the size of an off-manifold kick across eleven orders | null |
| **C4** | It *does* move when the trajectory cannot leave the manifold at all | **positive control** |

C4 is what makes the first three mean anything. Two null results on their own are just
"nothing happened."

## Setup

**No global default dtype.** The parent notebook pins `float64` at the top and forgets about
it. Here precision is the variable under test, so every tensor constructor takes `dtype=`
explicitly and nothing relies on `torch.get_default_dtype()`.

This matters more than it looks. A 0-dim tensor in PyTorch is *weak* for type promotion:
`torch.tensor(1e-6, dtype=torch.float64) * W` where `W` is `float32` returns **`float32`**,
silently, with no error. So the failure mode is not a loud dtype mismatch - it is a `float64`
scalar quietly demoted, or a `float32` factor quietly promoted, and a sweep that measures the
same precision twice under two names. Assert, don't assume.

In [1]:
import time

import torch
import numpy as np
import matplotlib.pyplot as plt
from mpmath import mp

In [2]:
# The system, unchanged from the parent notebook: docs/edge-of-stability.md section 1.
S_LIST = [1.0, 0.62, 0.38, 0.24, 0.15]   # target spectrum; s_max = 1 by construction
N      = len(S_LIST)
A0     = 1e-6                            # initial mode strength

# Thresholds implied by s_max = 1.
S_MAX      = max(S_LIST)
EDGE       = 1.0 / S_MAX                 # 1/s_max - fixed point loses stability
BLOWUP     = 2.0 / S_MAX                 # 2/s_max - scalar mode map escapes

# Sweep configuration. Fixed here rather than inline in the drivers so the whole
# experiment is legible in one cell.
BRACKET       = (0.9, 2.1)               # bisection bracket; must straddle the crossing
TOL           = 1e-5                     # -> ceil(log2(1.2/1e-5)) = 17 bisection steps
ESCAPE_BUDGET = 20_000                   # steps per run; trap 1 - the boundary moves with it
ESCAPE_MULT   = 1e6                      # escape = loss > this x its OWN initial loss
SEEDS         = (0, 1, 2)                # trap 7 - U, V are random, so report a range
DELTAS        = (1e-12, 1e-10, 1e-8, 1e-6, 1e-4, 1e-2, 1e-1)

# The three precision arms. mpmath is not a dtype and is handled separately.
DTYPES = {"float32": torch.float32, "float64": torch.float64}
MP_PREC = 64                             # bits of mantissa

plt.rcParams.update({"figure.dpi": 130, "font.size": 9})

mp.prec = MP_PREC
print(f"{'arm':<10} {'machine eps':>12}")
for name, dt in DTYPES.items():
    print(f"{name:<10} {torch.finfo(dt).eps:>12.3e}")
print(f"{'mpmath':<10} {float(mp.eps):>12.3e}   (prec={mp.prec})")
print(f"\nbisection steps at tol={TOL:g}: "
      f"{int(np.ceil(np.log2((BRACKET[1] - BRACKET[0]) / TOL)))}")

arm         machine eps
float32       1.192e-07
float64       2.220e-16
mpmath        1.084e-19   (prec=64)

bisection steps at tol=1e-05: 17


## Two thresholds, not one

The loss is $L = \tfrac12 \lVert \Sigma - W_2 W_1 \rVert_F^2$. Under balanced aligned
initialisation the modes decouple and each one follows a scalar problem
$\tfrac12 (s - uv)^2$. At its minimum $u = v = \sqrt{s}$ the Hessian is

$$H = \begin{bmatrix} s & s \\ s & s \end{bmatrix}, \qquad \lambda = 2s,\; 0$$

so the sharpness is $\lambda_{\max} = 2 s_{\max}$, and the textbook descent condition
$\eta < 2/\lambda_{\max}$ gives

$$\eta < \frac{1}{s_{\max}} = 1.0$$

That is the **edge of stability**: above it the fixed point is no longer attracting. It is
*not* where the iterate blows up. The scalar map $a \leftarrow a\,(1 + \eta(s - a))^2$ stays
bounded until

$$\eta = \frac{2}{s_{\max}} = 2.0$$

The window $1 < \eta < 2$ is bounded but non-convergent - the edge-of-stability regime, and
the interesting part of this notebook. `docs/deep-linear-dynamics.md` §7 records these two
being conflated; they land at 1 and 2 here only because $s_{\max} = 1$.

**The claim under test is that the full 5x5 network leaves that window early** - that it
escapes well below 2.0, because rounding knocks it off the manifold on which the modes
decouple in the first place, and off that manifold the scalar map no longer governs it.

## The runs

Every measurement below is one call to `train_variant` with different knobs, so its contract
has to be right before anything else is written.

### The problem instance is fixed across precision arms

$U$ and $V$ are drawn **once in `float64`** from the seed, then cast to the working type.
Every arm therefore solves the same problem, and C2's null result isolates the arithmetic and
nothing else.

The alternative - running the QR itself in `float32` for the `float32` arm - is the more
literal reading of the protocol's trap 4, but it makes $U$ and $V$ differ between arms by
$\sim10^{-7}$, so C2 would vary the instance *and* the arithmetic together and the seed
spread would have to absorb the difference. Trap 4 is about the *iteration* being dtype-pure;
that still holds here, and every operation after the cast runs in the working type.

Casting `float64` to `mpf` at `prec=64` is **exact** - 53 mantissa bits into 64 - so the
extended arm starts from bit-identical $U$, $V$ and differs only in how it iterates.

### $\Sigma$ is assembled after the cast

`Sigma = U @ diag(S) @ V.T` computed in `float64` and then cast down is **not** the same
matrix as the cast factors multiplied together in `float32`. One is the correctly-rounded
`float32` image of the exact product; the other is the `float32` product of rounded factors.
Pick the second - it is what the arm's own arithmetic would produce.

### The kick direction is fixed per seed; only its magnitude varies with $\delta$

C3 sweeps $\delta$ over eleven orders and claims no trend. If the random direction is redrawn
at each $\delta$, the sweep varies direction and magnitude together and the flatness is
noisier than it needs to be for no gain. Draw one direction matrix per seed and scale it.

*(Not in the protocol - an addition, and worth recording as one.)*

In [3]:
from typing import NamedTuple


class Run(NamedTuple):
    """What one training run reports back.

    Fixed here because `escaped`, `escape_threshold` and the C1 panel all consume it;
    changing the shape later means changing three call sites.

    loss0       loss at step 0, before any update. The escape test is scale-free
                relative to THIS, never an absolute cutoff (trap 1).
    final_loss  loss at the last step actually taken. inf/nan is a legal value.
    escaped_at  step index at which the escape condition first tripped, or None if
                the run survived the budget.
    steps_taken how many steps actually ran. Less than `steps` when it bailed early.
    losses      full trajectory, or None unless record=True was asked for. C1's
                algebraic-decay check at eta=1 needs it; nothing else does, and
                keeping 20k floats per run for 33 bisections is pointless.
    """
    loss0: float
    final_loss: float
    escaped_at: int | None
    steps_taken: int
    losses: list[float] | None = None


# ========================================================== 1 of 3 - start here
def make_problem(seed, dtype):
    """U, V, Sigma for one seed, in one working dtype.

    The top half of `train()` in deep_linear_dynamics.ipynb, plus a cast.

    Contract:
      - U, V are ALWAYS drawn in float64 from `seed`, then cast to `dtype`. Same
        instance in every arm, so C2 varies arithmetic alone.
      - Sigma is assembled from the CAST factors, in `dtype` - not assembled in
        float64 and cast afterwards. Those are different matrices.
      - returns (U, V, Sigma), every one of dtype `dtype`.
    """
    torch.manual_seed(seed)
    U = torch.linalg.qr(torch.randn(N, N, dtype=torch.float64))[0].to(dtype)
    V = torch.linalg.qr(torch.randn(N, N, dtype=torch.float64))[0].to(dtype)

    Sigma = U @ torch.diag(torch.tensor(S_LIST, dtype=dtype)) @ V.T

    tol = 32 * torch.finfo(dtype).eps

    assert torch.allclose(U @ U.T, torch.eye(N, dtype=dtype),rtol=tol, atol=tol)
    assert torch.allclose(torch.linalg.svdvals(Sigma), torch.tensor(S_LIST, dtype=dtype), rtol=tol, atol=tol)
    return U, V, Sigma


# check: U @ U.T is I, svdvals(Sigma) is S, and all three carry `dtype`.


# ========================================================== 2 of 3 - two minutes
def kick_direction(seed, dtype):
    """One fixed off-manifold direction per seed, scaled later by delta * sqrt(a0).

    Contract:
      - drawn from its OWN torch.Generator, never the global RNG. If it consumes the
        global stream, asking for a kick changes which U, V you get, so delta=0 and
        delta=1e-12 become different problems - the exact artefact C3 exists to rule
        out. (A fresh Generator seeded with n gives the same stream as
        torch.manual_seed(n), so this costs you nothing.)
      - returns (D1, D2), shapes matching W1 and W2, dtype `dtype`.
      - generic: it must break balance AND alignment. A direction that preserves
        W2.T @ W2 - W1 @ W1.T stays on the manifold and C3 measures nothing (trap 6).
    """

    g = torch.Generator().manual_seed(seed)

    D1 = torch.randn(N, N, dtype=dtype, generator=g)
    D2 = torch.randn(N, N, dtype=dtype, generator=g)

    return (D1, D2)


# check: make_problem(0, f64) returns the same U whether or not this ran first.


def is_escaped(loss, loss0):
    """The escape rule. Lives in exactly one place because trap 1 says the
    threshold depends on how this is defined.

    Scale-free: non-finite, or a large multiple of THIS run's own initial loss.
    An absolute cutoff would bake the answer in.

    Section 3's `escaped(run)` is then just `run.escaped_at is not None` - the
    rule is applied here, per step, so the bisection never re-derives it.
    """
    return not torch.isfinite(loss) or loss > ESCAPE_MULT * loss0


# ========================================================== 3 of 3 - the real one
def train_variant(eta, seed, dtype, delta, steps, identity_init=False, record=False):
    """One run of the deep linear system with the knobs this sweep needs.

    The bottom half of `train()` with four changes, marked A-D below. Everything
    else is the loop you already wrote and verified against the closed form.

    Contract:
      - dtype controls the ENTIRE pipeline: Sigma, U, V, both factors, the update.
        Not just the factors (trap 4).
      - delta adds an off-manifold perturbation to both factors at init, scaled by
        sqrt(a0). delta=0 means the unperturbed baseline run.
      - identity_init=True uses U = V = I instead of random orthogonal.
      - returns a Run, not a plot.
      - `steps` is the NUMBER OF UPDATES. Hold that convention: the budget-doubling
        check compares 20k against 40k and is meaningless if they disagree by one.
      - with record=True, losses[0] is loss0, so len(losses) == steps_taken + 1.
    """
    s = torch.tensor(S_LIST, dtype=dtype)

    # 1. the problem. identity_init uses torch.eye: the exactness of those
    #    off-diagonal zeros IS the C4 control (trap 5).
    if identity_init:
        U = V = torch.eye(N, dtype=dtype)
        Sigma = torch.diag(s)              # U diag(s) V.T collapses when U = V = I
    else:
        U, V, Sigma = make_problem(seed, dtype)

    # 2. balanced init.
    #    CHANGE A. dtype= is explicit here. train() wrote torch.sqrt(torch.tensor(a0)),
    #    a 0-dim tensor, which is WEAK for type promotion - it takes its dtype from
    #    whatever it multiplies. Harmless in the parent, where the global default was
    #    float64. Here it is trap 4 itself.
    root = torch.tensor(A0, dtype=dtype).sqrt()
    W1 = root * V.T
    W2 = root * U

    # 3. CHANGE B. the off-manifold kick. identity_init together with delta > 0 is a
    #    legal and NECESSARY combination - it is half of C4 - so the identity branch
    #    above must not skip this.
    if delta:
        D1, D2 = kick_direction(seed, dtype)
        W1 = W1 + delta * root * D1
        W2 = W2 + delta * root * D2

    # 4. loss0, from the initialised and kicked factors, before any update.
    E = W2 @ W1 - Sigma
    loss = 0.5 * (E ** 2).sum()
    loss0 = loss
    losses = [loss.item()] if record else None

    # 5. the loop.
    #    CHANGE C. no svdvals - the loss is all that is needed, and svdvals is ~20%
    #              of the whole sweep (measured: 74 vs 62 us/step).
    #    CHANGE D. test escape every step and return the moment it trips.
    for k in range(1, steps + 1):
        g1, g2 = W2.T @ E, E @ W1.T        # both gradients from the SAME E
        W1 = W1 - eta * g1
        W2 = W2 - eta * g2
        E = W2 @ W1 - Sigma
        loss = 0.5 * (E ** 2).sum()
        if record:
            losses.append(loss.item())
        if is_escaped(loss, loss0):
            return Run(loss0.item(), loss.item(), k, k, losses)

    # 6. survived the budget.
    return Run(loss0.item(), loss.item(), None, steps, losses)


# check: eta=0.01, seed=0, float64, delta=0, steps=6000 reproduces the parent
# notebook exactly - loss0 = 0.804448, final_loss = 5.84223e-08, which is the
# `final loss : 5.84e-08` in its committed output. If that passes the loop is right,
# and you can trust it at eta ~ 1.27 where nothing checks it.


### Init checks — these are not decoration

Three of the four claims rest on an assertion rather than on a plot. If the assertion is
missing, the claim is missing.

**Trap 4 — `float32` must be `float32` all the way through.** Check `.dtype` on $\Sigma$,
$U$, $V$, both factors *and* one update output. Not just the inputs: the whole point is that
promotion happens mid-loop. Diagnostic — if the `float32` and `float64` thresholds later
agree to *more* digits than the seed spread does, suspect an upcast before believing the null
result. Two arms measuring the same arithmetic agree suspiciously well.

**Trap 5 — the identity control must be asserted, not assumed.** With $U = V = I$ every
factor stays exactly diagonal, so the run carries the same rounding as every other run but
cannot be pushed off the manifold. Check the off-diagonal entries are **exactly** zero:

```
offdiag.abs().max() == 0        # not < 1e-30, not torch.allclose
```

and check it *after* the run, not just at init — a single nonzero off-diagonal anywhere in
20,000 steps and C4 establishes nothing. That assertion **is** the control.

**Trap 6 — the kick must actually leave the manifold.** After perturbing, the balance
residual must be nonzero:

```
(W2.T @ W2 - W1 @ W1.T).abs().max() > 0
```

**Representability.** $\sqrt{a_0} = 10^{-3}$, so a $\delta$ kick has size $\delta \times
10^{-3}$ against entries of $10^{-3}$:

| | ulp at $10^{-3}$ | $\delta = 10^{-15}$ kick $= 10^{-18}$ | $\delta = 10^{-12}$ kick $= 10^{-15}$ |
|---|---|---|---|
| `float32` | $1.2 \times 10^{-10}$ | **vanishes** | **vanishes** |
| `float64` | $2.2 \times 10^{-19}$ | 4.5 ulp — survives | ~4500 ulp — survives |

So C4's kicked control runs at `float64` or better, and C3's whole $\delta$ range is
`float64`-only. Assert the kick changed the tensor rather than assuming it — a perturbation
that rounds away is indistinguishable from `delta=0`, and it would look like a *finding*.

**Inherited from the parent notebook:** both factors step from the *same* $E$. Stepping
$W_2$ first and using the updated value to form $\nabla_{W_1}L$ is a different algorithm.

In [4]:
# Init checks. Run once per arm before trusting anything downstream.
#
#   1. dtype purity      - Sigma, U, V, W1, W2 and one update output, in every dtype
#   2. balanced init     - W2.T @ W2 == W1 @ W1.T == a0 * I at delta=0
#                          (pin rtol/atol: the default atol=1e-8 is 1e4x a0 itself and
#                           would accept any a0 within ~1%, as the parent notebook found)
#   3. identity is exact - off-diagonals of W1, W2 exactly 0 under identity_init
#   4. kick lands        - balance residual nonzero, and W changed, for every delta
#   5. kick survives     - the perturbed tensor differs from the unperturbed one at the
#                          smallest delta used in that arm
